<a href="https://colab.research.google.com/github/davidrpugh/introduction-to-deep-learning/blob/master/notebooks/01a-logistic-regression-from-scratch-with-pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Logistic Regression from Scratch with PyTorch

In [ ]:
import numpy as np
import torch

from sklearn import compose, datasets, linear_model, metrics, model_selection
from sklearn import pipeline, preprocessing

## Loading the data

In [ ]:
covtype_dataset = datasets.fetch_covtype(
    as_frame=True
)

In [ ]:
print(covtype_dataset["DESCR"])

In [ ]:
covtype_features_df = covtype_dataset["data"]
covtype_target_df = (
    covtype_dataset.get("target")
                   .to_frame()
)

In [ ]:
covtype_features_df.info()

In [ ]:
_ = (
    covtype_target_df.loc[:, "Cover_Type"]
                     .value_counts()
                     .sort_index()
                     .plot(kind="bar")
)

## Preparing the data

### Train/Val split

In [ ]:
RANDOM_STATE = np.random.RandomState(42)


train_features_df, val_features_df, train_target_df, val_target_df = (
    model_selection.train_test_split(
        covtype_features_df,
        covtype_target_df,
        test_size=0.20,
        shuffle=True,
        stratify=covtype_target_df,
        random_state=RANDOM_STATE
    )
)


In [ ]:
train_features_df.info()

In [ ]:
val_features_df.info()

### Features and target prepartion

In [ ]:
def array_to_tensor(arr, dtype=torch.float32):
  return torch.tensor(arr, dtype=dtype)


prepare_covtype_features = pipeline.make_pipeline(
    compose.make_column_transformer(
        (
            "passthrough",
            compose.make_column_selector(
                pattern="^Wilderness_Area_|^Soil_Type_"
            )
        ),
        force_int_remainder_cols=False,
        n_jobs=-1,
        remainder=preprocessing.QuantileTransformer(
            output_distribution="normal",
            random_state=RANDOM_STATE,
        )
    ),
    preprocessing.FunctionTransformer(
        func=array_to_tensor,
    )
)

prepare_covtype_target = pipeline.make_pipeline(
    preprocessing.OrdinalEncoder(
        categories=[
            [1, 2, 3, 4, 5, 6, 7]
        ],
    ),
    preprocessing.FunctionTransformer(
        func=array_to_tensor,
        kw_args={
            "dtype": torch.int64
        }
    ),
    preprocessing.FunctionTransformer(
        func=torch.squeeze,
    )
)



In [ ]:
X_train = prepare_covtype_features.fit_transform(train_features_df)
X_val = prepare_covtype_features.transform(val_features_df)


In [ ]:
print(X_train.shape)
print(X_val.shape)

In [ ]:
y_train = prepare_covtype_target.fit_transform(train_target_df)
y_val = prepare_covtype_target.transform(val_target_df)


In [ ]:
print(y_train.shape)
print(y_train.dtype)

print(y_val.shape)
print(y_val.dtype)

## Logistic Regression using PyTorch Tensors

### Initialize parameters

In [ ]:
prng = torch.manual_seed(42)

n_features = X_train.size(1)
n_classes = y_train.unique().size(0)

weights = torch.randn((n_classes, n_features), requires_grad=True)
bias = torch.zeros(1, n_classes, requires_grad=True)

In [ ]:
print(weights.shape)
print(bias.shape)

### Define our model and loss functions

In [ ]:
def model_fn(X):
    return X @ weights.T + bias


def softmax_activation_fn(y_pred):
    return torch.exp(y_pred) / torch.sum(torch.exp(y_pred), dim=1, keepdim=True)


def log_softmax_activation_fn(y_pred):
    return torch.log(softmax_activation_fn(y_pred))


def neg_log_likelihood_loss_fn(log_probas_pred, y_true):
    batch_indicies = torch.arange(y_true.size(0))
    target_indices = y_true
    return -torch.mean(log_probas_pred[batch_indicies, target_indices])


def cross_entropy_loss_fn(y_pred, y_true):
    log_probas_pred = log_softmax_activation_fn(y_pred)
    return neg_log_likelihood_loss_fn(log_probas_pred, y_true)



### Training using full batch gradient descent

In [ ]:
learning_rate = 0.5
n_epochs = 100

for epoch in range(n_epochs):
    # forward pass
    y_pred = model_fn(X_train)
    train_loss = cross_entropy_loss_fn(y_pred, y_train)

    # backward pass
    train_loss.backward()

    # gradient descent step
    with torch.no_grad():
        bias -= learning_rate * bias.grad
        weights -= learning_rate * weights.grad
        bias.grad.zero_()
        weights.grad.zero_()

    # evaluate using the validation data
    with torch.no_grad():
        y_pred = model_fn(X_val)
        val_loss = cross_entropy_loss_fn(y_pred, y_val)

    print(f"Epoch {epoch + 1}/{n_epochs}, Training Loss: {train_loss.item(): .4f}, Val Loss: {val_loss.item(): .4f}")



## Logistic Regression using the Neural Network API

In [ ]:
from torch import nn, optim

### The `nn.Linear` module

* The [`nn.Linear`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html) module implements a linear transformation:

$$ y = XW^T + b $$

* It’s a subclass of [`nn.Module`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html), the base class for all neural network components.
* Think of modules as math LEGO bricks: modules are the building blocks for complex models.

In [ ]:
_ = torch.manual_seed(42)

# logistic regression only has a single layer
covtype_model = nn.Linear(
    in_features=n_features,
    out_features=n_classes,
    bias=True,
)

#### Parameters

Each `nn.Linear` module includes:

* Weight matrix, $W$, with shape = `(out_features, in_features)`
* Bias vector, $b$, with one term per output neuron
* Parameters are instances of [`nn.Parameter`](https://docs.pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html) which is a subclass of Tensor with `requires_grad=True`.
* Parameters are randomly initialized from a uniform distribution (lots more on this later!).


In [ ]:
print(covtype_model.weight)

In [ ]:
print(covtype_model.bias)

#### Accessing Model Parameters

Access model parameters via:

1. `model.parameters()` → iterator over parameters
2. `model.named_parameters()` → iterator over `(name, value)` pairs



In [ ]:
for (name, param) in covtype_model.named_parameters():
    print(f"{name}: {param}")

#### A model's forward pass creates a computation graph

* Calling a module like a function (e.g., `model(X)`) internally invokes its `forward()` method.
* Autograd automatically tracks operations to build the computation graph.
* Output tensors contain a `grad_fn` attribute for gradient tracking.

In [ ]:
print(covtype_model(X_train[:5, :]))

### Loss functions and optimizers

* Define an appropriate loss function or criterion for multi-class classification. Since we are implementing Logistic Regression we should use [`nn.CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html).
* Define an optimizer: For now we will just use vanilla [`optim.SGD`](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html)

In [ ]:
# rare classes get larger weight when calculating the loss
n_samples = y_train.size(0)
weights = n_samples / (n_classes * torch.bincount(y_train))
normalized_weights = weights / weights.sum()

print(weights)
print(normalized_weights)

In [ ]:
cross_entropy_loss = nn.CrossEntropyLoss(
    weight=normalized_weights,
)

# to create an optimizer we must provide parameters and a learning rate!
sgd = optim.SGD(
    covtype_model.parameters(),
    lr=1e0
)

### Define a training loop

In [ ]:
def train(
    model_fn,
    criterion,
    optimizer,
    X_train,
    y_train,
    X_val,
    y_val,
    n_epochs
    ):

    for epoch in range(n_epochs):
        # forward pass
        y_pred = model_fn(X_train)
        train_loss = criterion(y_pred, y_train)

        # backward pass
        train_loss.backward()

        # gradient descent step
        optimizer.step()
        optimizer.zero_grad()

        # evaluate using the validation data
        with torch.no_grad():
            y_pred = model_fn(X_val)
            val_loss = criterion(y_pred, y_val)

        print(f"Epoch {epoch + 1}/{n_epochs}, Training Loss: {train_loss.item(): .4f}, Val Loss: {val_loss.item(): .4f}")


In [ ]:
train(
    covtype_model,
    cross_entropy_loss,
    sgd,
    X_train,
    y_train,
    X_val,
    y_val,
    n_epochs=100
)

### Exercise (Optional):

Train an unregularized logistic regression model with Scikit-Learn using the [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) class from the `linear_model` package. Compare the results with your results above.

In [ ]:
# INSERT YOUR CODE HERE!

### Solution:

In [ ]:
logistic_regression = pipeline.make_pipeline(
    prepare_covtype_features[:-1],
    linear_model.LogisticRegression(
        class_weight="balanced",
        max_iter=1000
    )
)

# train a logistic regression model
train_target = train_target_df.loc[:, "Cover_Type"]
_ = logistic_regression.fit(train_features_df, train_target)

# evaluate the model using the training set
train_prediction = logistic_regression.predict_proba(train_features_df)
train_log_loss = metrics.log_loss(
    train_target,
    train_prediction
)

# evaluate the model using the validation set
val_target = val_target_df.loc[:, "Cover_Type"]
val_prediction = logistic_regression.predict_proba(val_features_df)
val_log_loss = metrics.log_loss(
    val_target,
    val_prediction
)

print(f"Training Loss: {train_log_loss: .4f}, Val Loss: {val_log_loss: .4f}")


## Exercise:

Load the breast cancer dataset using the code in the cell below. Prepare the data and then train a logistic regression model using PyTorch Neural Network API.

In [ ]:
breast_cancer_dataset = datasets.load_breast_cancer(
    as_frame=True,
)

In [ ]:
print(breast_cancer_dataset["DESCR"])

In [ ]:
breast_cancer_features_df = breast_cancer_dataset["data"]
breast_cancer_target = breast_cancer_dataset["target"]

In [ ]:
breast_cancer_features_df.info()

In [ ]:
breast_cancer_features_df.describe()

In [ ]:
_ = breast_cancer_target.hist()

In [ ]:
# INSERT YOUR CODE HERE!

### Solution:

In [ ]:
train_features_df, val_features_df, train_target, val_target = (
    model_selection.train_test_split(
        breast_cancer_features_df,
        breast_cancer_target,
        random_state=RANDOM_STATE,
        shuffle=True,
        stratify=breast_cancer_target,
        test_size=0.20,
    )
)


In [ ]:
def dataframe_to_tensor(df, dtype=torch.float32):
    arr = df.to_numpy()
    return array_to_tensor(arr, dtype)


def series_to_tensor(s, dtype=torch.float32):
    arr = s.to_numpy()
    return array_to_tensor(arr, dtype)


n_samples, _ = train_features_df.shape
prepare_breast_cancer_features = pipeline.make_pipeline(
    preprocessing.QuantileTransformer(
      n_quantiles=n_samples,
      output_distribution="normal",
      random_state=RANDOM_STATE
    ),
    preprocessing.FunctionTransformer(
        func=array_to_tensor
    )
)

prepare_breast_cancer_target = pipeline.make_pipeline(
    preprocessing.FunctionTransformer(
        func=series_to_tensor,
        kw_args={
            "dtype": torch.int64
        }
    )
)

In [ ]:
X_train = prepare_breast_cancer_features.fit_transform(train_features_df)
X_val = prepare_breast_cancer_features.transform(val_features_df)


In [ ]:
print(X_train.shape)
print(X_val.shape)

In [ ]:
y_train = prepare_breast_cancer_target.fit_transform(train_target)
y_val = prepare_breast_cancer_target.transform(val_target)


In [ ]:
print(y_train.shape)
print(y_val.shape)

In [ ]:
_ = torch.manual_seed(42)

# logistic regression only has a single layer
n_samples = X_train.size(0)
n_features = X_train.size(1)
n_classes = y_train.unique().size(0)

breast_cancer_model = nn.Linear(
    in_features=n_features,
    out_features=n_classes,
    bias=True,
)

# continue using the cross-entropy loss
weights = n_samples / (n_classes * torch.bincount(y_train))
normalized_weights = weights / weights.sum()
cross_entropy_loss = nn.CrossEntropyLoss(
    weight=normalized_weights,
)

# define our optimizer
learning_rate = 1e-2
sgd = optim.SGD(
    breast_cancer_model.parameters(),
    lr=learning_rate
)

# train the model
train(
    breast_cancer_model,
    cross_entropy_loss,
    sgd,
    X_train,
    y_train,
    X_val,
    y_val,
    n_epochs=100
)

### Exercise (Optional):

Train an unregularized logistic regression model with Scikit-Learn using the [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) class from the `linear_model` package. Compare the results with your results above.

In [ ]:
# INSERT YOUR CODE HERE!

### Solution

In [ ]:
# train logisitic regression on the breast cancer data
logistic_regression = pipeline.make_pipeline(
    prepare_breast_cancer_features[:-1],
    linear_model.LogisticRegression(
        class_weight="balanced",
    )
)
_ = logistic_regression.fit(train_features_df, train_target)

# evaluate on the training set
train_prediction = logistic_regression.predict_proba(train_features_df)
train_log_loss = metrics.log_loss(
    train_target,
    train_prediction
)

# evaluate on the validation set
val_prediction = logistic_regression.predict_proba(val_features_df)
val_log_loss = metrics.log_loss(
    val_target,
    val_prediction
)

print(f"Training loss: {train_log_loss: .4f}, Validation loss: {val_log_loss: .4f}")